# 09 — LLM Fraud Investigation Reports

This notebook adds the **LLM investigation layer** on top of the anomaly-detection pipeline.

The LLM does **not** detect fraud and does **not** calculate anomaly scores.

Its role is limited to:
- reading a structured investigation packet produced by notebook 08
- summarizing the anomaly signals
- separating detector evidence from contextual evidence
- recommending analyst checks
- explicitly stating uncertainty and missing information
- producing a consistent structured investigation report

Pipeline:

`Anomaly detectors → explanation/context packet → LLM → analyst-facing report`

The ground-truth `fraud` label is intentionally excluded from LLM inputs.


## Design constraints

The LLM must never:

- reinterpret anomaly scores as probabilities
- claim that a transaction is confirmed fraud
- invent customer history or merchant behavior that is not in the packet
- use the hidden test label
- change the risk tier calculated upstream
- replace analyst review

This makes the LLM an **investigation assistant**, not the anomaly detector.


In [1]:
from pathlib import Path
import json
import os
from datetime import datetime, timezone

import pandas as pd


## Paths

In [2]:
RESULTS_PATH = Path("../results")
INVESTIGATION_PATH = RESULTS_PATH / "investigations"
LLM_REPORT_PATH = RESULTS_PATH / "llm_reports"

PACKETS_PATH = (
    INVESTIGATION_PATH
    / "top_100_investigation_packets.json"
)

LLM_REPORT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

print("Packets:", PACKETS_PATH)
print("Reports:", LLM_REPORT_PATH)


Packets: ..\results\investigations\top_100_investigation_packets.json
Reports: ..\results\llm_reports


## Load investigation packets

In [3]:
with open(
    PACKETS_PATH,
    "r",
    encoding="utf-8"
) as f:
    investigation_packets = json.load(f)

print(
    "Investigation packets:",
    len(investigation_packets)
)

investigation_packets[0]


Investigation packets: 100


{'transaction_id': 561747,
 'risk_tier': 'HIGH',
 'models_top_5pct': 3,
 'mean_anomaly_percentile': 0.9997202315621839,
 'transaction': {'step': 171,
  'customer': 'C1837321151',
  'merchant': 'M732195782',
  'category': 'es_travel',
  'amount': 1796.07,
  'age': '0',
  'gender': 'M'},
 'model_scores': {'isolation_forest': {'score': -0.021856840952361234,
   'percentile': 0.9998708761056233,
   'max_f1_alert': True,
   'high_recall_alert': True},
  'one_class_svm': {'score': 124.66078661261568,
   'percentile': 0.9994835044224933,
   'max_f1_alert': True,
   'high_recall_alert': True},
  'autoencoder': {'score': 0.037667468190193176,
   'percentile': 0.999806314158435,
   'max_f1_alert': True,
   'high_recall_alert': True}},
 'context': {'global_amount_percentile': 0.9993225112959239,
  'category_amount_percentile': 0.48268839103869654,
  'merchant_amount_percentile': 0.45363408521303256,
  'category_frequency': 0.0013096336759896936,
  'merchant_frequency': 0.0010642440666392826,
  'r

## Validate that no ground-truth label is sent to the LLM

In [4]:
def contains_forbidden_ground_truth(
    obj
):
    forbidden_keys = {
        "fraud",
        "ground_truth_fraud",
        "label",
        "target"
    }

    if isinstance(obj, dict):
        for key, value in obj.items():

            if str(key).lower() in forbidden_keys:
                return True

            if contains_forbidden_ground_truth(
                value
            ):
                return True

    elif isinstance(obj, list):
        return any(
            contains_forbidden_ground_truth(item)
            for item in obj
        )

    return False


for i, packet in enumerate(
    investigation_packets
):
    if contains_forbidden_ground_truth(
        packet
    ):
        raise ValueError(
            f"Ground-truth field detected "
            f"in packet {i}"
        )

print(
    "Validation passed: "
    "no ground-truth fraud label "
    "is present in LLM inputs."
)


Validation passed: no ground-truth fraud label is present in LLM inputs.


## Report schema

In [5]:
REPORT_SCHEMA = {
    "type": "object",
    "properties": {
        "transaction_id": {
            "type": "integer"
        },

        "risk_tier": {
            "type": "string",
            "enum": [
                "HIGH",
                "MEDIUM",
                "LOW"
            ]
        },

        "executive_summary": {
            "type": "string"
        },

        "model_evidence": {
            "type": "array",
            "items": {
                "type": "string"
            }
        },

        "contextual_evidence": {
            "type": "array",
            "items": {
                "type": "string"
            }
        },

        "autoencoder_evidence": {
            "type": "array",
            "items": {
                "type": "string"
            }
        },

        "model_consensus": {
            "type": "string"
        },

        "recommended_analyst_checks": {
            "type": "array",
            "items": {
                "type": "string"
            }
        },

        "limitations": {
            "type": "array",
            "items": {
                "type": "string"
            }
        },

        "conclusion": {
            "type": "string"
        }
    },

    "required": [
        "transaction_id",
        "risk_tier",
        "executive_summary",
        "model_evidence",
        "contextual_evidence",
        "autoencoder_evidence",
        "model_consensus",
        "recommended_analyst_checks",
        "limitations",
        "conclusion"
    ],

    "additionalProperties": False
}

REPORT_SCHEMA


{'type': 'object',
 'properties': {'transaction_id': {'type': 'integer'},
  'risk_tier': {'type': 'string', 'enum': ['HIGH', 'MEDIUM', 'LOW']},
  'executive_summary': {'type': 'string'},
  'model_evidence': {'type': 'array', 'items': {'type': 'string'}},
  'contextual_evidence': {'type': 'array', 'items': {'type': 'string'}},
  'autoencoder_evidence': {'type': 'array', 'items': {'type': 'string'}},
  'model_consensus': {'type': 'string'},
  'recommended_analyst_checks': {'type': 'array', 'items': {'type': 'string'}},
  'limitations': {'type': 'array', 'items': {'type': 'string'}},
  'conclusion': {'type': 'string'}},
 'required': ['transaction_id',
  'risk_tier',
  'executive_summary',
  'model_evidence',
  'contextual_evidence',
  'autoencoder_evidence',
  'model_consensus',
  'recommended_analyst_checks',
  'limitations',
  'conclusion'],
 'additionalProperties': False}

## System instructions

In [6]:
SYSTEM_PROMPT = '''
You are a fraud-investigation assistant.

You receive a structured anomaly-investigation packet that was produced
by upstream machine-learning detectors.

Your role is to summarize and contextualize the evidence for a human analyst.

STRICT RULES:

1. Do not decide whether the transaction is truly fraudulent.
2. Never state or imply that an anomaly score is a probability of fraud.
3. Do not invent facts, customer history, merchant history, locations,
   identities, account information, or external events.
4. Use only information contained in the supplied packet.
5. The upstream risk_tier must be copied exactly. Do not change it.
6. Clearly distinguish:
   - model evidence,
   - contextual/historical evidence,
   - Autoencoder reconstruction evidence.
7. If models disagree, explicitly state the disagreement.
8. Recommended checks must be actions for a human analyst, not claims that
   those checks already occurred.
9. State important limitations, including missing behavioral or identity
   context where relevant.
10. The conclusion must remain investigative. Use language such as
    "warrants review", "anomalous", or "requires verification";
    never use "confirmed fraud".
11. Be concise and factual.
'''


## Deterministic local dry-run report

Before making any API call, we can validate the end-to-end pipeline locally.

This deterministic renderer is **not an LLM**. It creates a simple report from the packet so that file paths, data structures and exports can be tested without cost.


In [7]:
def local_report_from_packet(
    packet
):
    model_evidence = []

    for model_name, info in packet[
        "model_scores"
    ].items():

        model_evidence.append(
            (
                f"{model_name}: anomaly percentile "
                f"{info['percentile']:.3f}; "
                f"max-F1 alert={info['max_f1_alert']}; "
                f"high-recall alert={info['high_recall_alert']}."
            )
        )

    contextual_evidence = list(
        packet["context"]["reasons"]
    )

    autoencoder_evidence = []

    for feature in packet[
        "autoencoder_top_reconstruction_features"
    ]:
        autoencoder_evidence.append(
            (
                f"{feature['feature']}: "
                f"squared reconstruction error "
                f"{feature['squared_error']:.6f}."
            )
        )

    agreement = packet[
        "models_top_5pct"
    ]

    return {
        "transaction_id":
            int(packet["transaction_id"]),

        "risk_tier":
            packet["risk_tier"],

        "executive_summary":
            (
                f"Transaction routed as "
                f"{packet['risk_tier']} risk based on "
                f"upstream anomaly detection and "
                f"investigation-routing rules."
            ),

        "model_evidence":
            model_evidence,

        "contextual_evidence":
            contextual_evidence,

        "autoencoder_evidence":
            autoencoder_evidence,

        "model_consensus":
            (
                f"{agreement} of 3 models place "
                f"the transaction in their top 5% "
                f"anomaly region."
            ),

        "recommended_analyst_checks": [
            "Verify the transaction with available account/customer context.",
            "Review nearby transactions for unusual temporal or merchant patterns.",
            "Check whether the amount and merchant behavior are expected for the account."
        ],

        "limitations": [
            "Anomaly scores are not fraud probabilities.",
            "The packet contains limited behavioral history.",
            "The report does not establish ground-truth fraud."
        ],

        "conclusion":
            (
                "The transaction is anomalous according to "
                "the supplied detector evidence and warrants "
                "review according to its assigned risk tier."
            )
    }


In [8]:
dry_run_report = local_report_from_packet(
    investigation_packets[0]
)

print(
    json.dumps(
        dry_run_report,
        indent=2
    )
)


{
  "transaction_id": 561747,
  "risk_tier": "HIGH",
  "executive_summary": "Transaction routed as HIGH risk based on upstream anomaly detection and investigation-routing rules.",
  "model_evidence": [
    "isolation_forest: anomaly percentile 1.000; max-F1 alert=True; high-recall alert=True.",
    "one_class_svm: anomaly percentile 0.999; max-F1 alert=True; high-recall alert=True.",
    "autoencoder: anomaly percentile 1.000; max-F1 alert=True; high-recall alert=True."
  ],
  "contextual_evidence": [
    "Amount is above the 99th percentile of training transactions.",
    "Transaction category is rare in training history.",
    "Merchant is rare in training history.",
    "All three anomaly detectors place the transaction in their top 5% most anomalous region.",
    "One-Class SVM triggered its high-confidence alert threshold."
  ],
  "autoencoder_evidence": [
    "cat__age_0: squared reconstruction error 0.887110.",
    "cat__merchant_M732195782: squared reconstruction error 0.875075

## Render a report as Markdown

In [9]:
def report_to_markdown(
    report
):
    def bullets(items):
        return "\n".join(
            f"- {item}"
            for item in items
        )

    return f'''# Transaction Investigation Report

**Transaction ID:** {report["transaction_id"]}  
**Risk tier:** {report["risk_tier"]}

## Executive summary

{report["executive_summary"]}

## Model evidence

{bullets(report["model_evidence"])}

## Contextual evidence

{bullets(report["contextual_evidence"])}

## Autoencoder reconstruction evidence

{bullets(report["autoencoder_evidence"])}

## Model consensus

{report["model_consensus"]}

## Recommended analyst checks

{bullets(report["recommended_analyst_checks"])}

## Limitations

{bullets(report["limitations"])}

## Conclusion

{report["conclusion"]}
'''


In [10]:
print(
    report_to_markdown(
        dry_run_report
    )
)


# Transaction Investigation Report

**Transaction ID:** 561747  
**Risk tier:** HIGH

## Executive summary

Transaction routed as HIGH risk based on upstream anomaly detection and investigation-routing rules.

## Model evidence

- isolation_forest: anomaly percentile 1.000; max-F1 alert=True; high-recall alert=True.
- one_class_svm: anomaly percentile 0.999; max-F1 alert=True; high-recall alert=True.
- autoencoder: anomaly percentile 1.000; max-F1 alert=True; high-recall alert=True.

## Contextual evidence

- Amount is above the 99th percentile of training transactions.
- Transaction category is rare in training history.
- Merchant is rare in training history.
- All three anomaly detectors place the transaction in their top 5% most anomalous region.
- One-Class SVM triggered its high-confidence alert threshold.

## Autoencoder reconstruction evidence

- cat__age_0: squared reconstruction error 0.887110.
- cat__merchant_M732195782: squared reconstruction error 0.875075.
- cat__category_

## Optional: OpenAI API setup

The following section is optional.

Install/update the official Python SDK once in your environment:

```bash
pip install -U openai
```

Set your API key as an environment variable rather than hard-coding it in the notebook.

PowerShell example:

```powershell
$env:OPENAI_API_KEY="your-key"
```

For a persistent Windows user environment variable you can configure it outside the notebook.

The notebook defaults to a cost-sensitive text model, but the model name can be overridden with the `OPENAI_MODEL` environment variable.


In [11]:
OPENAI_AVAILABLE = False

try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
    print("OpenAI SDK available.")
except ImportError:
    print(
        "OpenAI SDK not installed. "
        "The local dry-run workflow still works."
    )


OpenAI SDK not installed. The local dry-run workflow still works.


In [12]:
MODEL = os.getenv(
    "OPENAI_MODEL",
    "gpt-5.6-luna"
)

HAS_API_KEY = bool(
    os.getenv("OPENAI_API_KEY")
)

print("Model:", MODEL)
print("API key configured:", HAS_API_KEY)


Model: gpt-5.6-luna
API key configured: False


## LLM report generator

In [13]:
def generate_llm_report(
    packet,
    model=MODEL
):
    if not OPENAI_AVAILABLE:
        raise RuntimeError(
            "Install the OpenAI Python SDK first."
        )

    if not os.getenv(
        "OPENAI_API_KEY"
    ):
        raise RuntimeError(
            "OPENAI_API_KEY is not configured."
        )

    if contains_forbidden_ground_truth(
        packet
    ):
        raise ValueError(
            "Ground-truth label detected in packet."
        )

    client = OpenAI()

    response = client.responses.create(
        model=model,

        instructions=SYSTEM_PROMPT,

        input=(
            "Create an analyst-facing investigation "
            "report from this packet. "
            "Use only the supplied evidence.\n\n"
            + json.dumps(
                packet,
                ensure_ascii=False
            )
        ),

        text={
            "format": {
                "type": "json_schema",
                "name": "fraud_investigation_report",
                "strict": True,
                "schema": REPORT_SCHEMA
            }
        },

        store=False
    )

    report = json.loads(
        response.output_text
    )

    # Defensive check:
    # risk tier must remain an upstream decision.
    if (
        report["risk_tier"]
        != packet["risk_tier"]
    ):
        raise ValueError(
            "LLM changed the upstream risk tier."
        )

    if (
        report["transaction_id"]
        != packet["transaction_id"]
    ):
        raise ValueError(
            "LLM changed the transaction ID."
        )

    return report


## Generate one LLM report

In [14]:
# This cell makes an API call only when a key is configured.

if OPENAI_AVAILABLE and HAS_API_KEY:

    llm_report = generate_llm_report(
        investigation_packets[0]
    )

    print(
        json.dumps(
            llm_report,
            indent=2,
            ensure_ascii=False
        )
    )

else:

    llm_report = dry_run_report

    print(
        "No API call made. "
        "Using deterministic dry-run report."
    )


No API call made. Using deterministic dry-run report.


## Display the analyst-facing report

In [15]:
print(
    report_to_markdown(
        llm_report
    )
)


# Transaction Investigation Report

**Transaction ID:** 561747  
**Risk tier:** HIGH

## Executive summary

Transaction routed as HIGH risk based on upstream anomaly detection and investigation-routing rules.

## Model evidence

- isolation_forest: anomaly percentile 1.000; max-F1 alert=True; high-recall alert=True.
- one_class_svm: anomaly percentile 0.999; max-F1 alert=True; high-recall alert=True.
- autoencoder: anomaly percentile 1.000; max-F1 alert=True; high-recall alert=True.

## Contextual evidence

- Amount is above the 99th percentile of training transactions.
- Transaction category is rare in training history.
- Merchant is rare in training history.
- All three anomaly detectors place the transaction in their top 5% most anomalous region.
- One-Class SVM triggered its high-confidence alert threshold.

## Autoencoder reconstruction evidence

- cat__age_0: squared reconstruction error 0.887110.
- cat__merchant_M732195782: squared reconstruction error 0.875075.
- cat__category_

## Save one report

In [16]:
transaction_id = llm_report[
    "transaction_id"
]

json_output = (
    LLM_REPORT_PATH
    / f"transaction_{transaction_id}_report.json"
)

markdown_output = (
    LLM_REPORT_PATH
    / f"transaction_{transaction_id}_report.md"
)

with open(
    json_output,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        llm_report,
        f,
        indent=2,
        ensure_ascii=False
    )

with open(
    markdown_output,
    "w",
    encoding="utf-8"
) as f:
    f.write(
        report_to_markdown(
            llm_report
        )
    )

print("Saved:", json_output)
print("Saved:", markdown_output)


Saved: ..\results\llm_reports\transaction_561747_report.json
Saved: ..\results\llm_reports\transaction_561747_report.md


## Batch generation helper

For portfolio development, start with a very small batch.

There is no need to send all 100 packets to an API while iterating on the prompt.


In [17]:
def generate_report_batch(
    packets,
    limit=5,
    use_llm=True
):
    reports = []

    selected_packets = packets[
        :limit
    ]

    for i, packet in enumerate(
        selected_packets,
        start=1
    ):
        print(
            f"Processing {i}/{len(selected_packets)} "
            f"- transaction {packet['transaction_id']}"
        )

        if (
            use_llm
            and OPENAI_AVAILABLE
            and HAS_API_KEY
        ):
            report = generate_llm_report(
                packet
            )

        else:
            report = local_report_from_packet(
                packet
            )

        reports.append(
            report
        )

    return reports


In [18]:
# Safe default: no API calls.
#
# Set USE_LLM = True only when you intentionally
# want to generate API-backed reports.

USE_LLM = False
BATCH_SIZE = 5

batch_reports = generate_report_batch(
    investigation_packets,
    limit=BATCH_SIZE,
    use_llm=USE_LLM
)

len(batch_reports)


Processing 1/5 - transaction 561747
Processing 2/5 - transaction 549529
Processing 3/5 - transaction 504787
Processing 4/5 - transaction 551523
Processing 5/5 - transaction 491188


5

## Export batch reports as JSONL

In [19]:
batch_jsonl_path = (
    LLM_REPORT_PATH
    / "investigation_reports.jsonl"
)

with open(
    batch_jsonl_path,
    "w",
    encoding="utf-8"
) as f:

    for report in batch_reports:
        f.write(
            json.dumps(
                report,
                ensure_ascii=False
            )
            + "\n"
        )

print("Saved:", batch_jsonl_path)


Saved: ..\results\llm_reports\investigation_reports.jsonl


## Export a compact analyst table

In [20]:
report_summary = pd.DataFrame([
    {
        "transaction_id":
            report["transaction_id"],

        "risk_tier":
            report["risk_tier"],

        "executive_summary":
            report["executive_summary"],

        "model_consensus":
            report["model_consensus"],

        "conclusion":
            report["conclusion"]
    }

    for report in batch_reports
])

report_summary


,transaction_id,risk_tier,executive_summary,model_consensus,conclusion
0,561747,HIGH,Transaction routed as HIGH risk based on upstr...,3 of 3 models place the transaction in their t...,The transaction is anomalous according to the ...
1,549529,HIGH,Transaction routed as HIGH risk based on upstr...,3 of 3 models place the transaction in their t...,The transaction is anomalous according to the ...
2,504787,HIGH,Transaction routed as HIGH risk based on upstr...,3 of 3 models place the transaction in their t...,The transaction is anomalous according to the ...
3,551523,HIGH,Transaction routed as HIGH risk based on upstr...,3 of 3 models place the transaction in their t...,The transaction is anomalous according to the ...
4,491188,HIGH,Transaction routed as HIGH risk based on upstr...,3 of 3 models place the transaction in their t...,The transaction is anomalous according to the ...


In [21]:
report_summary_path = (
    LLM_REPORT_PATH
    / "investigation_report_summary.csv"
)

report_summary.to_csv(
    report_summary_path,
    index=False
)

print("Saved:", report_summary_path)


Saved: ..\results\llm_reports\investigation_report_summary.csv


## Simple report-quality checks

In [22]:
def validate_report(
    report,
    source_packet
):
    issues = []

    required_keys = set(
        REPORT_SCHEMA["required"]
    )

    missing = required_keys.difference(
        report.keys()
    )

    if missing:
        issues.append(
            f"Missing fields: {sorted(missing)}"
        )

    if (
        report.get("transaction_id")
        != source_packet["transaction_id"]
    ):
        issues.append(
            "Transaction ID mismatch."
        )

    if (
        report.get("risk_tier")
        != source_packet["risk_tier"]
    ):
        issues.append(
            "Risk tier mismatch."
        )

    text = json.dumps(
        report
    ).lower()

    forbidden_claims = [
        "confirmed fraud",
        "definitely fraudulent",
        "fraud probability"
    ]

    for phrase in forbidden_claims:
        if phrase in text:
            issues.append(
                f"Forbidden wording detected: {phrase}"
            )

    return issues


In [23]:
quality_rows = []

for packet, report in zip(
    investigation_packets[:len(batch_reports)],
    batch_reports
):
    issues = validate_report(
        report,
        packet
    )

    quality_rows.append({
        "transaction_id":
            packet["transaction_id"],
        "valid":
            len(issues) == 0,
        "issues":
            " | ".join(issues)
    })

quality_results = pd.DataFrame(
    quality_rows
)

quality_results


,transaction_id,valid,issues
0,561747,True,
1,549529,True,
2,504787,True,
3,551523,True,
4,491188,True,


## Prompt versioning

In [24]:
PROMPT_VERSION = "1.0"

prompt_metadata = {
    "prompt_version":
        PROMPT_VERSION,

    "model":
        MODEL,

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "report_schema":
        REPORT_SCHEMA,

    "system_prompt":
        SYSTEM_PROMPT
}

metadata_path = (
    LLM_REPORT_PATH
    / "prompt_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        prompt_metadata,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved:", metadata_path)


Saved: ..\results\llm_reports\prompt_metadata.json


## Final architecture

The project now separates responsibilities cleanly:

### Detection layer
- Isolation Forest
- One-Class SVM
- Autoencoder

Produces anomaly scores and operational alerts.

### Explainability layer
- score percentiles
- model agreement
- historical amount rarity
- merchant/category context
- Autoencoder reconstruction contributions

Produces structured investigation packets.

### LLM layer
- summarizes supplied evidence
- identifies model agreement/disagreement
- proposes analyst verification steps
- communicates limitations
- generates a standardized report

The LLM does **not**:
- train the anomaly detector
- calculate anomaly scores
- see the fraud ground-truth label
- declare confirmed fraud

This separation is central to the project's design.


## Next step

The next engineering step is to move reusable logic out of notebooks and into `src/`.

A practical structure is:

```text
src/
├── preprocessing.py
├── scoring.py
├── explainability.py
├── investigation.py
└── reporting.py
```

After that, we can build a small end-to-end CLI or application that accepts a transaction, scores it, builds an investigation packet, and produces an analyst report.
